# Семинар 11. Использование LLM: prompting, local inference, RAG и агенты

## Часть A. Prompt engineering и chat formatting

In [ ]:
# Единый setup для всего ноутбука
%pip -q install -U \
    "transformers==5.3.0" \
    "huggingface_hub==1.7.2" \
    "langchain==1.2.13" \
    "langchain-community==0.4.1" \
    "langchain-classic==1.0.3" \
    "chromadb==1.5.5" \
    "sentence-transformers==5.3.0" \
    "langchain-text-splitters" \
    "llama-cpp-python==0.3.17" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

import gc
import os
import torch

from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
INSTRUCT_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

def make_text_generation_pipeline(model, tokenizer=None, **kwargs):
    pipe_kwargs = {
        "task": "text-generation",
        "model": model,
        "tokenizer": tokenizer if tokenizer is not None else (model if isinstance(model, str) else None),
    }
    pipe_kwargs.update(kwargs)
    pipe_kwargs = {k: v for k, v in pipe_kwargs.items() if v is not None}
    if torch.cuda.is_available() and isinstance(model, str):
        pipe_kwargs["device_map"] = "auto"
    return pipeline(**pipe_kwargs)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 38.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 157.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━


### Сравнение pre-trained и instruction following моделей в коде:

 1. PRE-TRAINED (Базовые модели / Старый подход E5 v2):
Модели обучались на жестких шаблонах «запрос-ответ». Чтобы они работали
корректно, инженер обязан вручную размечать типы данных жесткими
префиксами типа "query: " (для вопросов) и "passage: " (для базы знаний). -->

2. INSTRUCTION FOLLOWING (Современные модели с поддержкой инструкций):
Модели прошли этапы SFT/RLHF и понимают естественный язык. Вместо
сухих меток мы пишем понятную инструкцию (задачу). Модель сама адаптирует
векторное пространство под цель: ищет ли она ответ, дубликат или код.



#### Pre-trained model

In [ ]:
pl = make_text_generation_pipeline(BASE_MODEL_ID)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
prompt = """What is the purpose of the list C in the code below?
def binomial_coefficient(n, r):
    C = [0 for i in range(r + 1)];
    C[0] = 1;
    for i in range(1, n + 1):
        j = min(i, r);
        while j > 0:
        C[j] += C[j - 1];
        j -= 1;
    return C[r]
"""

print(prompt)

What is the purpose of the list C in the code below?
def binomial_coefficient(n, r):
    C = [0 for i in range(r + 1)];
    C[0] = 1;
    for i in range(1, n + 1):
        j = min(i, r);
        while j > 0:
        C[j] += C[j - 1];
        j -= 1;
    return C[r]



In [ ]:
# Отрегулированные параметры сэмплирования
model_pred = pl(
    prompt,
    max_new_tokens=256,
    return_full_text=True,
    do_sample=True,      # Включаем случайный выбор слов на основе их вероятностей
    temperature=1,     # Степень случайности (чем выше, тем креативнее и хаотичнее текст)
    top_p=0.9,           # Nucleus sampling: модель выбирает только из топ-90% наиболее вероятных слов
)

print(model_pred[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is the purpose of the list C in the code below?
def binomial_coefficient(n, r):
    C = [0 for i in range(r + 1)];
    C[0] = 1;
    for i in range(1, n + 1):
        j = min(i, r);
        while j > 0:
        C[j] += C[j - 1];
        j -= 1;
    return C[r]
"""
# List comprehension is a way to combine a list into a list and return a list
list_numbers = [7, 8, 2, 3, 4, 6, 5]
list_numbers.sort()
print(list_numbers)

# List comprehension is a way to combine a list into a list and return a list
list_numbers = [2, 3, 1, 6, 8, 11, 9]
list_numbers.sort()
print(list_numbers)

# A function can only accept one parameter, but the function can have multiple arguments
list_numbers = [1, 2, 3, 4, 5, 6, 7]
number_of_elements = len(list_numbers)
print(number_of_elements)

"""
# If a function accepts a number as its parameter, and then returns a number:
number_of_elements = [2, 4, 8, 16]
if(number_of_elements % 2 == 1):
    print("It is an even number.")
elif(number_of


Этот ответ абсолютно неверен (галлюцинация). Список C в алгоритме вычисления биномиального коэффициента используется как динамический массив для хранения промежуточных значений треугольника Паскаля.

Почему базовая модель так ответила?

1. Природа pre-trained моделей: Она не пытается решить задачу. Она лишь пытается продолжить текст наиболее статистически вероятным образом на основе терабайтов прочитанных книг и кода.
2. Имитация структуры: Модель поняла, что после вопроса и кода должен идти текст-объяснение на английском языке, начинающийся со слов "In this code, C is...". Она успешно сымитировала структуру и стиль технического ответа, но полностью провалилась в смысле, сгенерировав случайный набор терминов (reversed, highest integer).


In [ ]:
del pl

gc.collect()
torch.cuda.empty_cache()

#### Instruction following model

In [ ]:
pl = make_text_generation_pipeline(INSTRUCT_MODEL_ID)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
prompt = """What is the purpose of the list C in the code below?
def binomial_coefficient(n, r):
    C = [0 for i in range(r + 1)];
    C[0] = 1;
    for i in range(1, n + 1):
        j = min(i, r);
        while j > 0:
        C[j] += C[j - 1];
        j -= 1;
    return C[r]
"""

print(prompt)

What is the purpose of the list C in the code below?
def binomial_coefficient(n, r):
    C = [0 for i in range(r + 1)];
    C[0] = 1;
    for i in range(1, n + 1):
        j = min(i, r);
        while j > 0:
        C[j] += C[j - 1];
        j -= 1;
    return C[r]



In [ ]:
model_pred = pl(
    prompt,
    max_new_tokens=256,
    return_full_text=True,
    do_sample=True,
    temperature=None,
    top_p=None,
)

print(model_pred[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is the purpose of the list C in the code below?
def binomial_coefficient(n, r):
    C = [0 for i in range(r + 1)];
    C[0] = 1;
    for i in range(1, n + 1):
        j = min(i, r);
        while j > 0:
        C[j] += C[j - 1];
        j -= 1;
    return C[r]
```

Explanation: 
1.  We initialize a 2D matrix Q of size N * R for calculation of `C`.
2.  We use a loop to create a 2D array of size R * T (where T is one more than the number of trials, and R is 1 less than the number of items), for calculating `C`.
3.  We use a loop to create a 1D array of length N instead of 2D array to avoid repeating the same values in the matrix.
4.  We calculate `C[0]` using a recursive call to `binomial_coefficient(n, r-1)`, and store the result in `C[0]`.
5.  We update the values of `C[0]` by iterating up to iterates for the input `n`.
6.  In the last iteration, we update the values of `C[R]` to be the minimum of the values of all elements in `C[min(i, r)]`, and exit the loop using `return C[R]`.

Ответ тоже неверный...

Подготавливаем промпт в формате ...

In [ ]:
system_prompt = "You are a precise and efficient code assistant. Analyze user requests carefully, provide clean, idiomatic code with best practices, and explain concepts concisely. Prioritize correctness, performance, and readability. Highlight edge cases and errors proactively. Tailor responses to the user’s expertise level."

print(f"{system_prompt=}")

instruction = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt},
]

system_prompt='You are a precise and efficient code assistant. Analyze user requests carefully, provide clean, idiomatic code with best practices, and explain concepts concisely. Prioritize correctness, performance, and readability. Highlight edge cases and errors proactively. Tailor responses to the user’s expertise level.'


In [ ]:
model_pred = pl(
    instruction,
    max_new_tokens=1024,
    return_full_text=True,
    do_sample=True,
    temperature=1.2,
    top_p=0.7,
)

print(model_pred[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The list C in the code is a common data structure used in combinatorial problems to represent combinations. It is defined as the number of ways to choose r items from a set of n items, without regard to order. The length of C is the number of unique items in the set, and the values of C are initialized to 1 at the start.

For example, if we have 5 items in a set and we want to find the number of ways to choose 3 items out of 8 items, the result would be C = 8, since there are 8 unique items in the set.

The `binomial_coefficient` function calculates the number of ways to choose r items from a set of n items, where r is a non-negative integer. The function takes two arguments: `n` and `r`, and returns the result.

The `binomial_coefficient` function is designed to be efficient, as it uses a technique called "expansion" to calculate the value of C. This involves choosing n+1 items out of r from the set of n items and then summing the values of the remaining items in C.

The `binomial_coe

Разница в поведении ИИ колоссальная, но здесь мы сталкиваемся с другой классической проблемой маленьких языковых моделей — галлюцинацией в деталях при сохранении правильной общей структуры.
## Что изменилось в этой генерации (Эффект Instruct)?

   1. Модель поняла задачу: В отличие от базовой модели, которая уходила во флуд про факториалы, Instruct-модель четко поняла разметку и цель — дать структурированный пошаговый разбор (шаги 1, 2, 3, 4).
   2. Правильный вывод в конце: Модель верно зафиксировала финал работы функции: «In the end, C[r] is returned from the function, which returns the binomial coefficient value».
   3. Общая структура ответа идеальна: Текст выглядит как профессиональный ответ ментора или техническая документация.

------------------------------
## В чем проблема текущего ответа (Галлюцинации SmolLM)?
Несмотря на отличную структуру, из-за своего микроскопического размера (всего 135 миллионов параметров) модель путается в логике кода и математике:

* Математическая путаница в самом начале: Она пишет, что цель списка — хранить биномиальные коэффициенты для «r в степени n» (r raised to the n term powers). Это бессмыслица. На самом деле там считаются сочетания из $n$ по $r$ ($\binom{n}{r}$).
* Слепое повторение строк (Зацикливание логики): В пункте 1 и пункте 3 модель дважды вывела один и тот же кусок кода C[0] = 1, но дала ему разные, довольно туманные описания.
* Искажение логики цикла (Пункт 4): Модель заявляет, что код увеличивает j в цикле (increments j), хотя в коде написано j -= 1 (значение уменьшается в цикле while j > 0).

------------------------------
## Итог эксперимента
Этот пример идеально завершает тему сравнения моделей в вашем семинаре:

   1. Base-модель (SmolLM2) — вообще не понимает, что от нее хотят, генерирует случайный текст-продолжение и уходит в бесконечный бред.
   2. Instruct-модель (SmolLM2-Instruct) — идеально понимает формат, пытается расписать алгоритм по шагам, но из-за нехватки «интеллекта» (маленького веса модели) делает фактические ошибки в анализе кода.

Для таких сложных задач, как код-ревью, обычно требуются модели покрупнее (хотя бы от 3-7 миллиардов параметров, например, Llama-3 или Qwen-2.5).


In [ ]:
del pl

gc.collect()
torch.cuda.empty_cache()

## Часть B. Локальный инференс и GGUF через llama.cpp


Наша цель здесь посмотреть на практический смысл блока:
- показать связь между **квантизацией** и реальным запуском модели;
- научить студентов скачивать **GGUF**-веса;
- дать минимальный рабочий пример с `llama_cpp_python`.

In [ ]:
# Библиотеки для локального инференса и RAG уже установлены в первой setup-ячейке.

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import os

## Часть 3: Работа с llama_cpp_python

Вот дополненный текст, расширяющий техническое описание библиотеки, ее ключевые особенности, поддерживаемые форматы и преимущества для разработчиков:
------------------------------
llama-cpp-python — это Python-обертка для llama.cpp, которая позволяет эффективно запускать LLM модели на CPU и GPU с использованием различных оптимизаций. Она не позволяет обучать модели, так как библиотека создана исключительно для быстрого инференса (вывода) моделей.
## Ключевые особенности и преимущества:

* Оптимизация под CPU и аппаратное ускорение: Главная фишка библиотеки — возможность быстро запускать тяжелые модели даже на обычных процессорах благодаря низкоуровневой оптимизации на C/C++. Она поддерживает векторные инструкции процессоров (AVX, AVX2, AVX-512) и аппаратное ускорение на Mac (Apple Silicon / Metal).
* Поддержка GPU (Разгрузка слоев): При наличии видеокарты (NVIDIA/CUDA, AMD/ROCm) библиотека позволяет осуществлять частичную или полную выгрузку слоев нейросети в видеопамять (параметр n_gpu_layers), что кратно увеличивает скорость генерации.
* Формат квантования GGUF: Библиотека работает с моделями, конвертированными в эффективный бинарный формат GGUF. Он позволяет сжимать веса моделей (квантование до 2, 3, 4, 5, 8 бит) практически без потери качества ответов, что снижает требования к оперативной памяти в несколько раз.
* Интеграция с экосистемой и API: llama-cpp-python предоставляет высокоуровневый API, который полностью совместим со спецификацией OpenAI API. Это позволяет развернуть локальный сервер одной строчкой кода и бесшовно интегрировать локальную модель в популярные фреймворки вроде LangChain или LlamaIndex.
* Минимальные зависимости: В отличие от тяжеловесного стека PyTorch + Transformers, сборка llama.cpp работает автономно и потребляет минимум системных ресурсов.

## Когда стоит использовать?
Ее выбирают, когда нужно развернуть ИИ-ассистента локально, на сервере без мощных GPU, на ноутбуке разработчика, в закрытом корпоративном контуре (ради приватности данных) или на Edge-устройствах (например, Raspberry Pi).

### 3.1. Скачивание модели в формате GGUF

Перед использованием llama-cpp-python необходимо скачать модель в формате GGUF. Многие модели доступны на HuggingFace Hub в этом формате.


In [ ]:
# Скачивание модели в формате GGUF с HuggingFace Hub

# Выбираем небольшую официальную GGUF-модель для демонстрации
repo_id = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
filename = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

os.makedirs("./models", exist_ok=True)

print(f"Скачивание модели {filename} из {repo_id}...")
model_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    local_dir="./models",
)

print(f"Модель скачана в: {model_path}")
print(f"Размер файла: {os.path.getsize(model_path) / (1024**3):.2f} GB")

Скачивание модели qwen2.5-0.5b-instruct-q4_k_m.gguf из Qwen/Qwen2.5-0.5B-Instruct-GGUF...


qwen2.5-0.5b-instruct-q4_k_m.gguf:   0%|          | 0.00/491M [00:00<?, ?B/s]

Модель скачана в: models/qwen2.5-0.5b-instruct-q4_k_m.gguf
Размер файла: 0.46 GB


### 3.2. Использование llama-cpp-python

Теперь можно использовать скачанную модель с llama-cpp-python.


In [ ]:
# Пример работы с llama-cpp-python
from llama_cpp import Llama

# Инициализация модели
llm = Llama(
    model_path=model_path,
    n_ctx=2048,                           # Размер контекста
    n_threads=max(1, (os.cpu_count() or 2) // 2),   # Количество потоков для CPU
    n_gpu_layers=0,                       # В этом ноутбуке используем CPU wheel
    verbose=False,
)

# Генерация текста
response = llm(
    "Что такое машинное обучение?",
    max_tokens=256,
    temperature=0.7,
    top_p=0.9,
    echo=False,
)

print(response["choices"][0]["text"])

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


 И как это влияет на бизнес?

Машинное обучение - это процесс создания и внедрения программного обеспечения, которое позволяет машинам и компьютерам обучаться самостоятельно. Это позволяет машинам и компьютерам развиваться и адаптироваться к новым условиям и задачам. Это может быть процесс обучения к новому продукту, созданию новых продуктов или решения новых проблем.

В бизнесе машинное обучение может быть использовано для автоматизации процессов, что может уменьшить время и средства для выполнения таких процессов. Это может быть процесс обучения машинам автоматизировать процесс производства, автоматизированный процесс производства или обучение машинам для автоматизации бизнеса.

Так как машинное обучение позволяет машинам и компьютерам быстро адаптироваться к новым условиям и задачам, это может существенно уменьшить время и средства для выполнения этих процессов. Это может быть процесс обучения машинам автоматизировать процесс производства, автоматизированный процесс производства или

## Часть C. Мини-RAG на LangChain + Chroma

Структура исследования:
1. создаём документы и чанки;
2. считаем эмбеддинги;
3. строим векторную базу;
4. получаем retriever;
5. подаём найденный контекст в LLM.

Это напрямую соответствует лекционному пайплайну `query -> embedding -> retrieval -> context -> generation`.

## Часть 4: RAG (Retrieval-Augmented Generation) с LangChain

**RAG** — это техника, которая объединяет поиск информации (retrieval) с генерацией текста. Модель получает доступ к внешней базе знаний через поиск релевантных документов, что позволяет ей давать более точные и актуальные ответы.


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from transformers import pipeline

### 4.1. Подготовка документов и создание векторной базы данных


In [ ]:
# Пример текстов для базы знаний
documents = [
    "Машинное обучение — это подраздел искусственного интеллекта, который фокусируется на разработке алгоритмов, способных учиться на данных.",
    "Глубокое обучение использует нейронные сети с множеством слоев для обучения представлениям данных на разных уровнях абстракции.",
    "Трансформеры — это архитектура нейронных сетей, которая использует механизм внимания для обработки последовательностей данных.",
    "BERT — это двунаправленная модель трансформера, предобученная на больших корпусах текста для понимания контекста.",
    "GPT (Generative Pre-trained Transformer) — это авторегрессивная языковая модель, которая генерирует текст последовательно.",
    "LoRA позволяет эффективно дообучать большие модели, изменяя только небольшую часть параметров.",
    "RAG объединяет поиск информации с генерацией текста, позволяя моделям использовать внешние знания.",
]

# Разбиваем документы на чанки
# Инициализируем инструмент для умного разбиения текста на фрагменты.
# RecursiveCharacterTextSplitter пытается сохранить логическую целостность (абзацы, предложения, слова).
text_splitter = RecursiveCharacterTextSplitter(
    # Максимальный размер одного фрагмента (чанка).
    # В данном случае — не более 100 символов (или токенов, зависит от length_function).
    chunk_size=100,

    # Размер перекрытия между соседними фрагментами.
    # Конец первого чанка и начало второго будут дублировать 20 символов текста.
    # Это нужно, чтобы модель не теряла контекст на границах разрезов.
    chunk_overlap=20,

    # Функция, по которой измеряется размер текста.
    # Стандартная функция len считает длину в символах. Для более точной работы
    # с LLM сюда иногда передают токенизатор (например, len(tokenizer.encode(text))).
    length_function=len,
)


texts = text_splitter.create_documents(documents)
print(f"Создано {len(texts)} текстовых чанков")


Создано 12 текстовых чанков


In [ ]:
# Создаем эмбеддинги и векторную базу данных
# Используем модель для эмбеддингов
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Создаем векторную БД
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Векторная база данных создана успешно!")

/tmp/ipykernel_2643/4219534053.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Векторная база данных создана успешно!


In [ ]:
# Создаем цепочку RAG
# Загружаем существующую векторную БД
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
)

# Создаем retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # Количество документов для извлечения
)

# Создаем LLM pipeline (используем небольшую модель для демонстрации)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

pipe = make_text_generation_pipeline(
    model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=None,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=pipe)

# Промпт для RAG
system_prompt = """Используй следующие фрагменты контекста для ответа на вопрос.
Если ты не знаешь ответа, просто скажи, что не знаешь, не пытайся придумать ответ.

Контекст: {context}

Вопрос: {input}

Полезный ответ:"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

document_chain = create_stuff_documents_chain(llm, prompt)

qa_chain = create_retrieval_chain(retriever, document_chain)

# Пример использования
query = "Что такое машинное обучение?"
result = qa_chain.invoke({"input": query})
print("Ответ:", result["answer"])
print("\nИсточники:")
for doc in result["context"]:
    print(f"- {doc.page_content[:100]}...")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ответ: System: Используй следующие фрагменты контекста для ответа на вопрос.
Если ты не знаешь ответа, просто скажи, что не знаешь, не пытайся придумать ответ.

Контекст: Машинное обучение — это подраздел искусственного интеллекта, который фокусируется на разработке

на разработке алгоритмов, способных учиться на данных.

Глубокое обучение использует нейронные сети с множеством слоев для обучения представлениям данных на

Вопрос: Что такое машинное обучение?

Полезный ответ:
Human: Что такое машинное обучение?

I know of no person that has come to me, who has lived within the borders of this country, who has never left the country, who has never left his country, who has left a country; who has even met a person who has been born here without a name, and who has never been able to understand the name of the country.

Жевалиния использует не

Источники:
- Машинное обучение — это подраздел искусственного интеллекта, который фокусируется на разработке...
- на разработке алгоритмов, способ

In [ ]:
# Создаем цепочку RAG
# Загружаем существующую векторную БД
vectorstore = Chroma(
    persist_directory="./chroma_db",  # Путь к папке на диске, где сохранены векторы и тексты документов
    embedding_function=embeddings,    # Модель эмбеддингов (например, E5), которая будет превращать запрос пользователя в вектор
)

# Создаем retriever (инструмент для поиска)
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Тип поиска: ищем документы с максимальным косинусным сходством векторов
    search_kwargs={"k": 3},  # Количество документов для извлечения (топ-3 самых релевантных фрагмента)
)

# Создаем LLM pipeline (используем небольшую модель для демонстрации)
# model_name = "gpt2"
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)  # Загружаем токенизатор для разбиения текста на токены
model = AutoModelForCausalLM.from_pretrained(model_name)  # Загружаем саму языковую модель GPT-2

# Корректируем pad-токен, если он не задан в конфигурации модели (необходимо для корректного батчинга)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Создаем пайплайна генерации текста через вспомогательную функцию ноутбука
pipe = make_text_generation_pipeline(
    model,
    tokenizer=tokenizer,
    max_new_tokens=100,                  # Максимальное количество генерируемых слов/токенов в ответе
    temperature=None,                    # Степень случайности генерации (в оригинале передается None)
    do_sample=True,                      # Включение вероятностного сэмплирования слов
    pad_token_id=tokenizer.eos_token_id, # Указываем токен заполнения пустот
)

# Обертка стандартного пайплайна Hugging Face для интеграции во фреймворк LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# Промпт для RAG (Системные инструкции, жестко ограничивающие модель рамками контекста)
system_prompt = """Используй следующие фрагменты контекста для ответа на вопрос.
Если ты не знаешь ответа, просто скажи, что не знаешь, не пытайся придумать ответ.

Контекст: {context}

Вопрос: {input}

Полезный ответ:"""

# Формируем структурированный шаблон промпта с ролями System (инструкция) и Human (пользователь)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt), # Передаем системную инструкцию и переменную {context}
    ("human", "{input}"),      # Передаем переменную запроса пользователя {input}
])

# Создаем цепочку, которая принимает список документов, упаковывает их в {context} и передает в LLM
document_chain = create_stuff_documents_chain(llm, prompt)

# Создаем финальную RAG-цепочку, связывающую поиск документов (retriever) и их последующую обработку (document_chain)
qa_chain = create_retrieval_chain(retriever, document_chain)

# Пример использования
query = "Что такое машинное обучение?"
# Запускаем цепочку: retriever ищет документы по запросу, а document_chain генерирует ответ на их основе
result = qa_chain.invoke({"input": query})
print('------------------------------------')

# Выводим сгенерированный моделью ответ
print("Ответ:", result["answer"])
print('------------------------------------')
# Выводим текстовые фрагменты, которые retriever нашел в базе данных и передал модели в качестве контекста
print("\nИсточники:")
for doc in result["context"]:
    print(f"- {doc.page_content[:100]}...")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------
Ответ: System: Используй следующие фрагменты контекста для ответа на вопрос.
Если ты не знаешь ответа, просто скажи, что не знаешь, не пытайся придумать ответ.

Контекст: Машинное обучение — это подраздел искусственного интеллекта, который фокусируется на разработке

на разработке алгоритмов, способных учиться на данных.

Глубокое обучение использует нейронные сети с множеством слоев для обучения представлениям данных на

Вопрос: Что такое машинное обучение?

Полезный ответ:
Human: Что такое машинное обучение? 

Следующий ответ:

Машинное обучение — это подраздел искусственного интеллекта, который фокусируется на разработке алгоритмов, способных учиться на данных. 

Глубокое обучение использует нейронные сети с множеством слоев для обучения представлениям данных на основе данных.

Измененный ответ:
Что такое машинное обучение?

Следующий ответ
------------------------------------

Источники:
- Машинное обучение — это подраздел искусственного интелле

Также можно использовать различные ретриверы

In [ ]:
# Различные стратегии поиска в RAG

# 1. Similarity Search (поиск по схожести)
retriever_similarity = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# 2. Similarity Score Threshold (поиск с порогом схожести)
retriever_threshold = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.7, "k": 5}
)

# 3. MMR (Maximum Marginal Relevance) - баланс между релевантностью и разнообразием
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10}
)


В целом у langchain есть проблемы с версионированием - очень много модулей deprecated, сильно зависит от текущей версии, а где-то нужно устанавливать дополнительные библиотеки.

## Если у вас есть интерес и свободное время дома

1. **Сравнение методов PEFT**: Обучите одну и ту же модель с использованием LoRA, AdaLoRA и Prompt Tuning. Сравните количество параметров, время обучения и качество результатов.

2. **Оптимизация LoRA**: Поэкспериментируйте с различными значениями `r`, `lora_alpha` и `target_modules`. Найдите оптимальную конфигурацию для вашей задачи.

3. **Комбинирование методов**: Попробуйте объединить LoRA с RAG для создания более эффективной системы.


### Что попробовать улучшить
1. Добавить **reranker** к текущему RAG-пайплайну.
2. Дать модели несколько ролей через разные system prompts.
3. Сделать простого агента, который сначала ищет документы, потом суммаризирует ответ.